# 🩻 InsightX — Baseline Models Evaluation

> ⚠️ **Research Demo Only** — This tool is not a certified medical device.
> Results are AI predictions for research purposes only and must not be used
> to guide clinical decisions. Always consult a qualified radiologist.

---

## Overview
This notebook evaluates three baseline model architectures for binary normal vs. abnormal finding prediction:
1. **Text-Only Baseline**: TF-IDF (top 1,000 features, negations preserved) + Logistic Regression.
2. **Image-Only Baseline**: Frozen MobileNetV2 Global Average Pooling embeddings (1,280-dim) + Logistic Regression.
3. **Multimodal Fused Baseline**: Horizontally concatenated features (2,280-dim) + Logistic Regression.

**Key Methodology Constraint**: All scalers and vocabulary vectorizers are fit strictly on the **train split only** to prevent data leakage into validation or test sets.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from insightx.text import clean_medical_text, get_medical_stopwords
from insightx.baselines import extract_or_load_image_features, build_tfidf_features

# Load the saved patient-level splits
splits_path = Path("../data/processed/splits.csv") if Path("../data/processed/splits.csv").exists() else Path("data/processed/splits.csv")
df = pd.read_csv(splits_path)
print(f"Loaded {len(df):,} records across {df['uid'].nunique():,} unique patients.")
print(df['split'].value_counts())

## Step 1: Image Feature Extraction (MobileNetV2)
We extract 1,280-dimensional visual feature vectors using a frozen MobileNetV2 pretrained on ImageNet (Global Average Pooling). Features are cached to `data/processed/image_features.npy`.

In [ ]:
# Load or extract cached image features
img_features = extract_or_load_image_features(df)
print(f"Image feature matrix shape: {img_features.shape}")

## Step 2: Clinical NLP Vectorization (TF-IDF)
We build a medical-grade TF-IDF vocabulary with 1,000 features on the clinical `findings` text.
Critically, **clinical negation tokens** (`no`, `not`, `none`, `without`, `negative`) are retained, and the vectorizer is **fit on the training split only**.

In [ ]:
train_mask = (df['split'] == 'train').to_numpy()
val_mask = (df['split'] == 'val').to_numpy()
test_mask = (df['split'] == 'test').to_numpy()

y_train = df.loc[train_mask, 'label'].to_numpy()
y_val = df.loc[val_mask, 'label'].to_numpy()
y_test = df.loc[test_mask, 'label'].to_numpy()

findings = df['findings'].fillna('').astype(str)
X_txt_train, X_txt_val, X_txt_test, tfidf_vec = build_tfidf_features(
    findings[train_mask], findings[val_mask], findings[test_mask], max_features=1000
)
print(f"Text feature train shape: {X_txt_train.shape}")
print(f"Text feature test shape:  {X_txt_test.shape}")

## Step 3 & 4: Train Baseline Models & Evaluate on Held-Out Test Set
We train three Logistic Regression models with `class_weight='balanced'`:
1. **Text-Only**
2. **Image-Only**
3. **Multimodal Fused** (horizontal stack)

Scalers are strictly fit on the training split only.

In [ ]:
X_img_train = img_features[train_mask]
X_img_test = img_features[test_mask]

X_fused_train = np.hstack((X_img_train, X_txt_train))
X_fused_test = np.hstack((X_img_test, X_txt_test))

modalities = {
    'Text-Only (Findings TF-IDF)': (X_txt_train, X_txt_test),
    'Image-Only (MobileNetV2 GAP)': (X_img_train, X_img_test),
    'Fused (Image + Text hstack)': (X_fused_train, X_fused_test),
}

results = []
for name, (X_tr, X_te) in modalities.items():
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)
    
    clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    clf.fit(X_tr_scaled, y_train)
    
    y_pred = clf.predict(X_te_scaled)
    y_proba = clf.predict_proba(X_te_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_score = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc*100:.2f}%',
        'F1-Score': f'{f1:.4f}',
        'ROC-AUC': f'{auc_score:.4f}',
    })

results_df = pd.DataFrame(results)
print('\n--- Test Split Baseline Evaluation ---')
print(results_df.to_string(index=False))


## 🔍 Clinical Interpretation & Justification for Production Architecture

### 1. High Performance of the Text-Only Baseline
The **text-only model scores noticeably higher** because in the Indiana University dataset, the MeSH and `Problems` annotations were indexed directly from the radiologist findings text. Consequently, the findings text shares direct linguistic overlap with the target ground truth.

### 2. Why the Production Served Model is Image-Based
In real clinical workflows, **no radiology report exists when a patient receives a chest radiograph**. An automated decision support tool that requires the report to assist the radiologist would create a circular dependency. InsightX is designed to assist clinicians at the point of care by directly analyzing the chest radiograph image and highlighting regions of interest with Grad-CAM.